# Simulating SPARC4 Simultaneous g,r,i,z Photometry — WASP-19
## *starmodel* educational notebook · Instrument: SPARC4 at OPD/LNA (Brazil)

### Overview
This notebook simulates the simultaneous multi-band transit photometry of
**WASP-19 b** as observed with **SPARC4** (Simultaneous Polarimeter and
Rapid Camera in 4 bands) at the **Observatório Pico dos Dias (OPD/LNA)**,
Brazil.

SPARC4 records four SDSS-like bands (g, r, i, z) simultaneously, making it
ideal for characterising the chromatic effects of limb darkening and stellar
activity on transit observations.

**Key references**
| Subject | Reference |
|---|---|
| WASP-19b discovery | Hebb et al. (2010), ApJ 708, 224 |
| Stellar parameters | Tregloan-Reed et al. (2013), MNRAS 428, 3671 |
| Spin-orbit alignment | Hellier et al. (2011), ApJ 730, L31 |
| Starspot constraints | Mancini et al. (2013), MNRAS 436, 2 |
| SPARC4 instrument | Martioli et al. (2023), PASP in prep |
| Chromatic LDs | Claret & Bloemen (2011), A&A 529, A75 |


### 1. SPARC4 at OPD/LNA

SPARC4 is an instrument developed by INPE and installed at the 1.6-m
Perkin-Elmer telescope at OPD/LNA in Brazópolis, MG, Brazil.

**Key specifications:**
| Feature | Value |
|---|---|
| Telescope | 1.6-m Perkin-Elmer, OPD/LNA |
| Mode | Simultaneous 4-channel imaging |
| Filters | SDSS g, r, i, z |
| Detector | 4 × iXon EMCCD 1024×1024 |
| Precision | ~1 mmag per transit |
| Field of view | 5.7′ × 5.7′ per channel |

The simultaneous g,r,i,z capability eliminates systematic errors from
time-variable atmospheric transmission, making colour-dependent effects
(limb darkening, starspot chromaticity) directly measurable.


### 2. WASP-19 System Parameters

WASP-19 b is an ultra-hot Jupiter with one of the shortest orbital periods
known ($P = 0.789$ days — only 19 hours!).

| Parameter | Value | Unit | Reference |
|---|---|---|---|
| $T_{\rm eff}$ | 5500 ± 100 | K | Hebb et al. (2010) |
| $R_\star$ | 1.004 ± 0.016 | $R_\odot$ | Tregloan-Reed et al. (2013) |
| $\log g_\star$ | 4.46 ± 0.01 | cgs | Tregloan-Reed et al. (2013) |
| $v \sin i$ | 4.0 ± 0.5 | km s$^{-1}$ | Hellier et al. (2011) |
| $[{\rm Fe/H}]$ | $+0.14 \pm 0.10$ | dex | Hebb et al. (2010) |
| $R_p/R_\star$ | 0.14361 ± 0.00054 | — | Tregloan-Reed et al. (2013) |
| $a/R_\star$ | 3.868 ± 0.006 | — | Tregloan-Reed et al. (2013) |
| $P_{\rm orb}$ | 0.7888399 | days | Hebb et al. (2010) |
| $i_{\rm orb}$ | 78.81 ± 0.31 | deg | Tregloan-Reed et al. (2013) |
| $\lambda$ | $4.6 \pm 5.2$ | deg | Hellier et al. (2011) |
| $T_0$ | 2455168.96801 | BJD | Tregloan-Reed et al. (2013) |

**Activity:** Mancini et al. (2013) detected starspot-crossing events in
multi-band photometry of WASP-19, finding spot temperature contrasts of
$\Delta T \approx 200$–$500$ K.


### 3. Chromatic Limb-Darkening and Transit Depth

A key observable for SPARC4 is the **wavelength-dependence of the transit
depth** arising from chromatic limb darkening.  The transit depth in band $x$ is:

$$\delta_x = \frac{\int_{\rm occulted} I_x(\mu)\, dA}{\int_{\rm disk} I_x(\mu)\, dA}$$

where $I_x(\mu)$ is the limb-darkened intensity in band $x$.  Since the
limb is darker in the blue (g) than in the red (z), the effective stellar
radius appears slightly larger in bluer bands, making the transit appear
**deeper in the blue**.

The chromatic signature of a **cool starspot** is the opposite: the spot
is relatively brighter in the red (cooler Planck), so the chromatic transit
residuals when the planet crosses a spot are redder.

**Differential colour light curve:**

$$\Delta(g{-}r)(t) = -2.5 \log_{10}\!\left(\frac{F_g(t)/F_{g,0}}{F_r(t)/F_{r,0}}\right)$$

Zero baseline = mean over the rotation cycle.


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from starmodel import (Star, TransitModel, OrbitalParameters,
                       StarSpot, Facula, GranulationField,
                       plot_transit_epoch)
from starmodel.stellar_variability import RotationSimulator

plt.style.use("dark_background")
FCOLOR = "#0d0d0d"
print("Imports OK")


In [ ]:
# ── WASP-19 system parameters ──────────────────────────────────────────────
# Hebb et al. (2010), Tregloan-Reed et al. (2013), Hellier et al. (2011)

WASP19 = dict(
    # Star
    T_eff     = 5500.,   # K
    R_star    = 1.004,   # R_sun
    v_sini    = 4.0,     # km/s
    inc_star  = 90.,     # assumed equator-on
    obliquity = 4.6,     # deg  (Hellier+2011)
    # Limb darkening in SDSS bands (Claret & Bloemen 2011, interpolated)
    # [a_quad, b_quad] per band
    ld_bands  = {
        "g": (0.61, 0.16),  # bluer → stronger limb darkening
        "r": (0.52, 0.20),
        "i": (0.44, 0.22),
        "z": (0.38, 0.24),
    },
    # Planet
    Rp_Rstar  = 0.14361,
    a_Rstar   = 3.868,
    P_orb     = 0.7888399, # days
    inc_orb   = 78.81,     # deg
    T0        = 0.0,
    lambda_RM = 4.6,       # deg
)
print("WASP-19 parameters loaded")
print(f"  Transit depth (geometric): {WASP19['Rp_Rstar']**2 * 100:.3f} %")


In [ ]:
# ── Build star with an active region ───────────────────────────────────────
# Mancini et al. (2013) report spot ΔT ~ 200–500 K; we use 350 K
# We include a representative facula co-located with the spot (active region)

wl = np.linspace(4000., 11000., 600)  # Covers full g,r,i,z range

star19 = (
    Star(n_theta=40, n_phi=80, name="WASP-19")
    .set_brightness(law="quadratic", coefficients={"a": 0.52, "b": 0.20})
    .set_rotation(v_eq=WASP19["v_sini"], inclination=WASP19["inc_star"],
                  obliquity=WASP19["obliquity"])
    .set_temperature_map(lambda e: WASP19["T_eff"])
)
star19.add_feature(GranulationField(n_cells=600, seed=3))
star19.add_feature(StarSpot(lat_deg=15., lon_deg=0., radius_deg=10., T_contrast=-350.))
star19.add_feature(Facula  (lat_deg=15., lon_deg=28., radius_deg=7., alpha=0.09))
star19.compute()
print(star19)


In [ ]:
# ── Transit simulation ─────────────────────────────────────────────────────
orbit19 = OrbitalParameters(
    period          = WASP19["P_orb"],
    t0              = WASP19["T0"],
    semi_major_axis = WASP19["a_Rstar"],
    inclination     = WASP19["inc_orb"],
    planet_radius   = WASP19["Rp_Rstar"],
    obliquity       = WASP19["lambda_RM"],
)

model19  = TransitModel(star19, orbit19)
result19 = model19.compute(n_times=500, compute_ccf=True)
print(result19.summary())


In [ ]:
# ── Plot: colour-dependent transit light curves ─────────────────────────────
# The SDSS g,r,i,z band light curves are automatically computed in TransitResult

t_h = (result19.times - orbit19.t0) * 24.

fig, axes = plt.subplots(2, 2, figsize=(13, 8), facecolor=FCOLOR)
axes = axes.ravel()
fig.suptitle("WASP-19 b — SPARC4 simultaneous g,r,i,z photometry
(Differential colour transit light curves)",
             fontsize=12, fontweight="bold", color="white")

band_colors = {"g−r": "#66aaff", "r−i": "#ffaa33", "i−z": "#cc55ff"}
band_labels = list(band_colors.keys())
band_col    = list(band_colors.values())

# Panel 0: broadband flux
axes[0].plot(t_h, (1 - result19.flux) * 1e6, color="white", lw=1.4)
axes[0].axhline(0, color="#555", lw=0.6, ls="--")
axes[0].set_title("Broadband transit depth", color="white")
axes[0].set_ylabel("ΔFlux (ppm)")

# Panels 1-3: differential colour LCs
for i, (name, col) in enumerate(band_colors.items()):
    ax = axes[i + 1]
    ax.plot(t_h, result19.color_lcs[i], color=col, lw=1.4)
    ax.axhline(0, color="#555", lw=0.7, ls="--")
    ax.set_title(f"Differential Δ({name}) — Spot crossing signature", color="white")
    ax.set_ylabel("Δcolour (mmag)")

for ax in axes:
    ax.set_facecolor("#111111")
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
    ax.tick_params(colors="#bbb"); ax.grid(True, alpha=0.12, color="#555")
    ax.set_xlabel("Time from mid-transit (h)")
    for ct_label, (col_c, ls) in {"T1":("#44ff88","-"),"T2":("#44ff88","--"),
                                    "T3":("#44ff88","--"),"T4":("#44ff88","-")}.items():
        tv = result19.contact_times.get(ct_label, float("nan"))
        if not np.isnan(tv):
            ax.axvline((tv - orbit19.t0)*24., color=col_c, lw=0.7, ls=ls, alpha=0.4)

plt.tight_layout(rect=[0,0,1,0.93])
plt.savefig("sparc4_wasp19_color_lcs.png", dpi=130, bbox_inches="tight", facecolor=FCOLOR)
plt.show()


In [ ]:
# ── Rotation variability over one WASP-19 "season" ─────────────────────────
# WASP-19 has P_rot ~ 10.5 days (Mancini+2013)
P_rot_wasp19 = 10.5   # days

sim19    = RotationSimulator(star19, P_rot_wasp19, n_phases_per_cycle=250)
res19_rot = sim19.run(n_cycles=2.)

fig2 = sim19.plot(res19_rot, time_unit="days")
fig2.suptitle(f"WASP-19 — SPARC4 rotation variability (P_rot = {P_rot_wasp19} d)",
              color="white", fontsize=11, fontweight="bold", y=0.978)
plt.savefig("sparc4_wasp19_rotation.png", dpi=130, bbox_inches="tight",
            facecolor=FCOLOR)
plt.show()
print(res19_rot.summary())


### Summary

This notebook demonstrated:

1. **Simultaneous g,r,i,z photometry** — the four colour channels show
   different transit depths due to chromatic limb darkening.

2. **Spot-crossing chromaticity** — when the planet occults a cool starspot,
   the flux excess (positive anomaly) is stronger in redder bands because
   the spot emits relatively more red flux than the photosphere.

3. **Rotation variability** — the rotating starspot produces a sinusoidal
   brightness modulation of ~10,000–40,000 ppm with a clear chromatic
   signature in the g−r, r−i, i−z colour indices.

**How SPARC4 data compares:**
SPARC4 achieves ~1 mmag photometric precision per data point in typical
observing conditions at OPD.  For a $V \approx 10$ star like WASP-19,
achieving the systematic noise floor requires careful detrending with
comparison stars in all four bands simultaneously.

The differential colour light curves (Δ(g−r), Δ(r−i), Δ(i−z)) are
powerful diagnostics for disentangling:
- Chromatic limb darkening (smooth, wavelength-dependent transit shape)
- Starspot contamination (sharp, localised colour anomalies)
- Atmospheric absorption (slope across all bands)
